In [1]:
import cv2
import numpy as np
import time

# ==========================================
# Video Path
# ==========================================

video_path = r"C:\Users\PC-LAB1\Desktop\New folder\day4\british_highway_traffic.mp4"

# ==========================================
# Open Video
# ==========================================

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Cannot open video")
    exit()

# ==========================================
# Video Information
# ==========================================

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

duration = frame_count / video_fps

print("Width:", width)
print("Height:", height)
print("Video FPS:", video_fps)
print("Frame Count:", frame_count)
print("Duration:", duration, "seconds")

# ==========================================
# Background Subtraction
# ==========================================

background = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

# ==========================================
# Kernel
# ==========================================

kernel = np.ones((5, 5), np.uint8)

# ==========================================
# FPS
# ==========================================

frame_number = 0
start_time = time.time()

# ==========================================
# Main Loop
# ==========================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # ======================================
    # 1 - Original
    # ======================================

    original = frame.copy()

    # ======================================
    # 2 - Grayscale
    # ======================================

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # ======================================
    # 3 - Gaussian Blur
    # ======================================

    blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    # ======================================
    # 4 - Canny
    # ======================================

    edges = cv2.Canny(
        blur,
        50,
        150
    )

    # ======================================
    # 5 - Threshold
    # ======================================

    _, binary = cv2.threshold(
        blur,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # ======================================
    # 6 - Morphology
    # ======================================

    morph = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel
    )

    # ======================================
    # 7 - Motion Detection
    # ======================================

    motion = background.apply(frame)

    _, motion = cv2.threshold(
        motion,
        200,
        255,
        cv2.THRESH_BINARY
    )

    motion = cv2.morphologyEx(
        motion,
        cv2.MORPH_OPEN,
        kernel
    )

    motion = cv2.dilate(
        motion,
        kernel,
        iterations=2
    )

    # ======================================
    # 8 - Contours
    # ======================================

    contours, _ = cv2.findContours(
        motion,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    detected = frame.copy()

    object_count = 0

    for contour in contours:

        area = cv2.contourArea(contour)

        # Ignore small objects
        if area < 800:
            continue

        x, y, w, h = cv2.boundingRect(contour)

        # Bounding Box
        cv2.rectangle(
            detected,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        object_count += 1

        # Object Number
        cv2.putText(
            detected,
            "Object " + str(object_count),
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

    # ======================================
    # 9 - FPS
    # ======================================

    elapsed_time = time.time() - start_time

    if elapsed_time > 0:
        fps = frame_number / elapsed_time
    else:
        fps = 0

    # ======================================
    # Information
    # ======================================

    cv2.putText(
        detected,
        "Frame: {}".format(frame_number),
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2
    )

    cv2.putText(
        detected,
        "FPS: {:.2f}".format(fps),
        (20, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

    cv2.putText(
        detected,
        "Objects: {}".format(object_count),
        (20, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    # ======================================
    # Resize All Frames
    # ======================================

    display_width = 400
    display_height = 250

    original_small = cv2.resize(
        original,
        (display_width, display_height)
    )

    gray_small = cv2.resize(
        gray,
        (display_width, display_height)
    )

    blur_small = cv2.resize(
        blur,
        (display_width, display_height)
    )

    edges_small = cv2.resize(
        edges,
        (display_width, display_height)
    )

    motion_small = cv2.resize(
        motion,
        (display_width, display_height)
    )

    detected_small = cv2.resize(
        detected,
        (display_width, display_height)
    )

    # ======================================
    # Convert Gray Images to BGR
    # ======================================

    gray_small = cv2.cvtColor(
        gray_small,
        cv2.COLOR_GRAY2BGR
    )

    blur_small = cv2.cvtColor(
        blur_small,
        cv2.COLOR_GRAY2BGR
    )

    edges_small = cv2.cvtColor(
        edges_small,
        cv2.COLOR_GRAY2BGR
    )

    motion_small = cv2.cvtColor(
        motion_small,
        cv2.COLOR_GRAY2BGR
    )

    # ======================================
    # Add Titles
    # ======================================

    cv2.putText(
        original_small,
        "1 - Original",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        gray_small,
        "2 - Grayscale",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        blur_small,
        "3 - Gaussian Blur",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        edges_small,
        "4 - Canny",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        motion_small,
        "5 - Motion",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        detected_small,
        "6 - Detection",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    # ======================================
    # Combine Frames
    # ======================================

    top_row = np.hstack((
        original_small,
        gray_small,
        blur_small
    ))

    bottom_row = np.hstack((
        edges_small,
        motion_small,
        detected_small
    ))

    # ======================================
    # Final Window
    # ======================================

    combined = np.vstack((
        top_row,
        bottom_row
    ))

    # ======================================
    # Display
    # ======================================

    cv2.imshow(
        "Day 04 - Real Time Video Processing",
        combined
    )

    # ======================================
    # Keyboard
    # ======================================

    key = cv2.waitKey(1) & 0xFF

    # Press q or ESC to stop
    if key == ord("q") or key == 27:
        break


# ==========================================
# Release
# ==========================================

cap.release()

cv2.destroyAllWindows()

print("Processing completed")
print("Total Frames:", frame_number)
print("Average FPS:", fps)

Width: 1364
Height: 768
Video FPS: 25.0
Frame Count: 1549
Duration: 61.96 seconds
Processing completed
Total Frames: 211
Average FPS: 28.128089038778544


In [ ]:
import cv2
import numpy as np
import time
import os

# ==========================================
# 1 - مسار الفيديو
# ==========================================

video_path = r'C:\Users\PC-LAB1\Desktop\MHB\day-4\dhaka_traffic.mp4'

# ==========================================
# 2 - مجلد حفظ جميع الإطارات
# ==========================================

save_folder = r"C:\Users\PC-LAB1\Desktop\New folder\day4\tracking_frames"

os.makedirs(save_folder, exist_ok=True)

# ==========================================
# 3 - فتح الفيديو
# ==========================================

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Cannot open video")
    exit()

# معلومات الفيديو
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = cap.get(cv2.CAP_PROP_FPS)

print("Video Width:", width)
print("Video Height:", height)
print("Video FPS:", video_fps)

# ==========================================
# 4 - Background Subtraction
# ==========================================

background = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

# Kernel للـ Morphology
kernel = np.ones((5, 5), np.uint8)

# ==========================================
# 5 - المتغيرات
# ==========================================

frame_number = 0

start_time = time.time()

# ==========================================
# 6 - قراءة الفيديو Frame by Frame
# ==========================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # نسخة من الصورة الأصلية
    original = frame.copy()

    # ==========================================
    # 7 - Gray
    # ==========================================

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # ==========================================
    # 8 - Gaussian Blur
    # ==========================================

    blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    # ==========================================
    # 9 - Canny Edge Detection
    # ==========================================

    edges = cv2.Canny(
        blur,
        50,
        150
    )

    # ==========================================
    # 10 - Motion Detection
    # ==========================================

    motion = background.apply(frame)

    # تحويل إلى Binary
    _, motion = cv2.threshold(
        motion,
        200,
        255,
        cv2.THRESH_BINARY
    )

    # إزالة النقاط الصغيرة
    motion = cv2.morphologyEx(
        motion,
        cv2.MORPH_OPEN,
        kernel
    )

    # تكبير المناطق المتحركة
    motion = cv2.dilate(
        motion,
        kernel,
        iterations=2
    )

    # ==========================================
    # 11 - استخراج Contours
    # ==========================================

    contours, _ = cv2.findContours(
        motion,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # نسخة للتتبع
    detected = frame.copy()

    object_count = 0

    # ==========================================
    # 12 - اكتشاف الأجسام
    # ==========================================

    for contour in contours:

        area = cv2.contourArea(contour)

        # تجاهل الأجسام الصغيرة
        if area < 800:
            continue

        # Bounding Box
        x, y, w, h = cv2.boundingRect(contour)

        # رسم المستطيل
        cv2.rectangle(
            detected,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        object_count += 1

        # كتابة رقم الجسم
        cv2.putText(
            detected,
            "Object " + str(object_count),
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

    # ==========================================
    # 13 - حساب FPS
    # ==========================================

    elapsed_time = time.time() - start_time

    if elapsed_time > 0:
        fps = frame_number / elapsed_time
    else:
        fps = 0

    # ==========================================
    # 14 - معلومات على الفيديو
    # ==========================================

    cv2.putText(
        detected,
        "Frame: {}".format(frame_number),
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2
    )

    cv2.putText(
        detected,
        "FPS: {:.2f}".format(fps),
        (20, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

    cv2.putText(
        detected,
        "Objects: {}".format(object_count),
        (20, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    # ==========================================
    # 15 - حفظ جميع الإطارات
    # ==========================================

    filename = os.path.join(
        save_folder,
        "tracking_frame_{:05d}.jpg".format(frame_number)
    )

    cv2.imwrite(
        filename,
        detected
    )

    # ==========================================
    # 16 - تصغير الصور للعرض
    # ==========================================

    display_width = 400
    display_height = 250

    original_small = cv2.resize(
        original,
        (display_width, display_height)
    )

    gray_small = cv2.resize(
        gray,
        (display_width, display_height)
    )

    blur_small = cv2.resize(
        blur,
        (display_width, display_height)
    )

    edges_small = cv2.resize(
        edges,
        (display_width, display_height)
    )

    motion_small = cv2.resize(
        motion,
        (display_width, display_height)
    )

    detected_small = cv2.resize(
        detected,
        (display_width, display_height)
    )

    # ==========================================
    # 17 - تحويل Gray إلى BGR
    # ==========================================

    gray_small = cv2.cvtColor(
        gray_small,
        cv2.COLOR_GRAY2BGR
    )

    blur_small = cv2.cvtColor(
        blur_small,
        cv2.COLOR_GRAY2BGR
    )

    edges_small = cv2.cvtColor(
        edges_small,
        cv2.COLOR_GRAY2BGR
    )

    motion_small = cv2.cvtColor(
        motion_small,
        cv2.COLOR_GRAY2BGR
    )

    # ==========================================
    # 18 - كتابة أسماء العمليات
    # ==========================================

    cv2.putText(
        original_small,
        "Original",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        gray_small,
        "Grayscale",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        blur_small,
        "Gaussian Blur",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        edges_small,
        "Canny",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        motion_small,
        "Motion",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.putText(
        detected_small,
        "Tracking / Detection",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    # ==========================================
    # 19 - الصف الأول
    # ==========================================

    top_row = np.hstack(
        (
            original_small,
            gray_small,
            blur_small
        )
    )

    # ==========================================
    # 20 - الصف الثاني
    # ==========================================

    bottom_row = np.hstack(
        (
            edges_small,
            motion_small,
            detected_small
        )
    )

    # ==========================================
    # 21 - دمج الصفوف
    # ==========================================

    combined = np.vstack(
        (
            top_row,
            bottom_row
        )
    )

    # ==========================================
    # 22 - عرض كل الإطارات في نافذة واحدة
    # ==========================================

    cv2.imshow(
        "Day 04 - Real Time Video Processing",
        combined
    )

    # ==========================================
    # 23 - التحكم
    # ==========================================

    key = cv2.waitKey(1) & 0xFF

    # Q للخروج
    if key == ord("q"):

        break

    # ESC للخروج
    if key == 27:

        break

# ==========================================
# 24 - إغلاق الفيديو
# ==========================================

cap.release()

cv2.destroyAllWindows()

print("--------------------------------")
print("Processing Finished")
print("Total Frames:", frame_number)
print("Saved Folder:")
print(save_folder)
print("--------------------------------")

Video Width: 640
Video Height: 360
Video FPS: 29.97002997002997
--------------------------------
Processing Finished
Total Frames: 342
Saved Folder:
C:\Users\PC-LAB1\Desktop\New folder\day4\tracking_frames
--------------------------------


In [13]:
import cv2
import numpy as np
import time

video_path = r"C:\Users\PC-LAB1\Desktop\New folder\day4\british_highway_traffic.mp4"

cap = cv2.VideoCapture(video_path)

background = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

kernel = np.ones((5, 5), np.uint8)

frame_number = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # ==========================================
    # بداية حساب زمن التراكن
    # ==========================================

    start = time.time()

    # Background Subtraction
    motion = background.apply(frame)

    # Threshold
    _, motion = cv2.threshold(
        motion,
        200,
        255,
        cv2.THRESH_BINARY
    )

    # Morphology
    motion = cv2.morphologyEx(
        motion,
        cv2.MORPH_OPEN,
        kernel
    )

    motion = cv2.dilate(
        motion,
        kernel,
        iterations=2
    )

    # Contours
    contours, _ = cv2.findContours(
        motion,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    detected = frame.copy()

    object_count = 0

    # Bounding Boxes
    for contour in contours:

        area = cv2.contourArea(contour)

        if area < 800:
            continue

        x, y, w, h = cv2.boundingRect(contour)

        cv2.rectangle(
            detected,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        object_count += 1

    # ==========================================
    # نهاية حساب زمن التراكن
    # ==========================================

    end = time.time()

    # زمن التراكن بالـ milliseconds
    tracking_time = (end - start) * 1000

    # FPS الخاص بالتراكن
    if tracking_time > 0:
        tracking_fps = 1000 / tracking_time
    else:
        tracking_fps = 0

    # ==========================================
    # عرض المعلومات
    # ==========================================

    cv2.putText(
        detected,
        "Frame: {}".format(frame_number),
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2
    )

    cv2.putText(
        detected,
        "Objects: {}".format(object_count),
        (20, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    cv2.putText(
        detected,
        "Tracking Time: {:.2f} ms".format(tracking_time),
        (20, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

    cv2.putText(
        detected,
        "Tracking FPS: {:.2f}".format(tracking_fps),
        (20, 125),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 0),
        2
    )

    cv2.imshow(
        "Tracking Performance",
        detected
    )

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [16]:
import cv2
import numpy as np
import time

video_path = r"C:\Users\PC-LAB1\Desktop\MHB\day-4\dhaka_traffic.mp4"

cap = cv2.VideoCapture(video_path)

background = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

kernel = np.ones((5, 5), np.uint8)

frame_number = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # ==========================================
    # بداية حساب زمن التراكن
    # ==========================================

    start = time.time()

    # Background Subtraction
    motion = background.apply(frame)

    # Threshold
    _, motion = cv2.threshold(
        motion,
        200,
        255,
        cv2.THRESH_BINARY
    )

    # Morphology
    motion = cv2.morphologyEx(
        motion,
        cv2.MORPH_OPEN,
        kernel
    )

    motion = cv2.dilate(
        motion,
        kernel,
        iterations=2
    )

    # Contours
    contours, _ = cv2.findContours(
        motion,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    detected = frame.copy()

    object_count = 0

    # Bounding Boxes
    for contour in contours:

        area = cv2.contourArea(contour)

        if area < 800:
            continue

        x, y, w, h = cv2.boundingRect(contour)

        cv2.rectangle(
            detected,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        object_count += 1

    # ==========================================
    # نهاية حساب زمن التراكن
    # ==========================================

    end = time.time()

    # زمن التراكن بالـ milliseconds
    tracking_time = (end - start) * 1000

    # FPS الخاص بالتراكن
    if tracking_time > 0:
        tracking_fps = 1000 / tracking_time
    else:
        tracking_fps = 0

    # ==========================================
    # عرض المعلومات
    # ==========================================

    cv2.putText(
        detected,
        "Frame: {}".format(frame_number),
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2
    )

    cv2.putText(
        detected,
        "Objects: {}".format(object_count),
        (20, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    cv2.putText(
        detected,
        "Tracking Time: {:.2f} ms".format(tracking_time),
        (20, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

    cv2.putText(
        detected,
        "Tracking FPS: {:.2f}".format(tracking_fps),
        (20, 125),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 0),
        2
    )

    cv2.imshow(
        "Tracking Performance",
        detected
    )

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [18]:
import cv2
import numpy as np
import time

video_path = r"C:\Users\PC-LAB1\Desktop\MHB\day-4\dhaka_traffic.mp4"

cap = cv2.VideoCapture(video_path)

background = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

kernel = np.ones((5, 5), np.uint8)

frame_number = 0

# ==========================================
# Stabilization state
# ==========================================
prev_gray = None


def stabilize_frame(prev_gray, curr_gray, curr_frame):
    """
    Estimates the camera's own motion between two consecutive
    frames (using tracked feature points) and warps the current
    frame to cancel that motion out. This way, whatever motion
    remains after background subtraction is mostly due to real
    moving objects, not the camera shake/pan.
    """

    prev_pts = cv2.goodFeaturesToTrack(
        prev_gray,
        maxCorners=200,
        qualityLevel=0.01,
        minDistance=30,
        blockSize=3
    )

    if prev_pts is None:
        return curr_frame

    curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray, curr_gray, prev_pts, None
    )

    if curr_pts is None or status is None:
        return curr_frame

    idx = status.flatten() == 1
    prev_pts = prev_pts[idx]
    curr_pts = curr_pts[idx]

    # Need a minimum number of matched points for a reliable estimate
    if len(prev_pts) < 6:
        return curr_frame

    m, _ = cv2.estimateAffinePartial2D(prev_pts, curr_pts)

    if m is None:
        return curr_frame

    # ---- Sanity check on the transform ----
    # For a handheld/walking camera the frame-to-frame motion can be
    # large; reject clearly invalid transforms (huge scale / huge
    # translation) instead of applying them and producing a mostly
    # black frame.
    scale = np.sqrt(m[0, 0] ** 2 + m[1, 0] ** 2)
    dx, dy = m[0, 2], m[1, 2]
    h, w = curr_frame.shape[:2]
    max_translation = 0.35 * np.hypot(w, h)  # allow up to 35% of the diagonal

    if scale < 0.7 or scale > 1.3:
        return curr_frame

    if np.hypot(dx, dy) > max_translation:
        return curr_frame

    # Invert the estimated camera-motion transform and apply it,
    # so the frame becomes "aligned" with the previous one.
    m_inv = cv2.invertAffineTransform(m)
    stabilized = cv2.warpAffine(
        curr_frame, m_inv, (w, h),
        borderMode=cv2.BORDER_REPLICATE
    )

    return stabilized


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # ==========================================
    # بداية حساب زمن التراكن
    # ==========================================

    start = time.time()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Stabilize the current frame against the previous one to
    # cancel out camera motion before feeding it to MOG2.
    if prev_gray is not None:
        frame = stabilize_frame(prev_gray, gray, frame)

    prev_gray = gray

    # Background Subtraction
    motion = background.apply(frame)

    # Threshold
    _, motion = cv2.threshold(
        motion,
        200,
        255,
        cv2.THRESH_BINARY
    )

    # Morphology
    motion = cv2.morphologyEx(
        motion,
        cv2.MORPH_OPEN,
        kernel
    )

    motion = cv2.dilate(
        motion,
        kernel,
        iterations=2
    )

    # Contours
    contours, _ = cv2.findContours(
        motion,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    detected = frame.copy()

    object_count = 0

    # Bounding Boxes
    for contour in contours:

        area = cv2.contourArea(contour)

        if area < 800:
            continue

        x, y, w, h = cv2.boundingRect(contour)

        cv2.rectangle(
            detected,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        object_count += 1

    # ==========================================
    # نهاية حساب زمن التراكن
    # ==========================================

    end = time.time()

    # زمن التراكن بالـ milliseconds
    tracking_time = (end - start) * 1000

    # FPS الخاص بالتراكن
    if tracking_time > 0:
        tracking_fps = 1000 / tracking_time
    else:
        tracking_fps = 0

    # ==========================================
    # عرض المعلومات
    # ==========================================

    cv2.putText(
        detected,
        "Frame: {}".format(frame_number),
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2
    )

    cv2.putText(
        detected,
        "Objects: {}".format(object_count),
        (20, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    cv2.putText(
        detected,
        "Tracking Time: {:.2f} ms".format(tracking_time),
        (20, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

    cv2.putText(
        detected,
        "Tracking FPS: {:.2f}".format(tracking_fps),
        (20, 125),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 0),
        2
    )

    cv2.imshow(
        "Tracking Performance",
        detected
    )

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()